# EXIOBASE import

### 1. Import required packages

In [ ]:
import bw2data as bd
import bw2io as bi
import pandas as pd

from functions.fun_exiobase_import import (
    get_location_from_name,
    production_as_datasetname,
    remove_empty_activities,
    remove_location_from_name,
    remove_old_ef,
    tweak3,
)
from functions.utils import Config

### 2. Select project

In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

### 3. Import biosphere database

The EXIOBASE import is linked to the standard Brightway `biosphere3` database.

In [ ]:
if Config.EXIOBASE_BIOSPHERE_NAME not in bd.databases:
    bi.create_default_biosphere3()

### 4. Create the EXIOBASE importer

EXIOBASE is imported from the SimaPro CSV file configured in `functions/utils.py`.

In [ ]:
importer = bi.SimaProBlockCSVImporter(
    Config.EXIOBASE_CSV_PATH,
    database_name=Config.EXIOBASE_DATABASE_NAME,
    biosphere_database_name=Config.EXIOBASE_BIOSPHERE_NAME,
    separate_products=False,
)

### 5. Apply standard and EXIOBASE-specific strategies

Brightway first applies the standard SimaPro import strategies. The additional helper strategies then clean EXIOBASE activity names, locations, old elementary flows, and production exchanges.

The detailed helper functions are available in `functions/fun_exiobase_import.py`.

In [ ]:
importer.apply_strategies()

importer.apply_strategy(get_location_from_name)
importer.apply_strategy(remove_location_from_name)
importer.apply_strategy(remove_old_ef)
importer.apply_strategy(production_as_datasetname)

# Check and normalize dataset units
print("Dataset units before normalization:")
print({ds["unit"] for ds in importer.data})

importer.apply_strategy(bi.strategies.generic.normalize_units)

print("Dataset units after normalization:")
print({ds["unit"] for ds in importer.data})

importer.apply_strategy(remove_empty_activities)

#### Check exchange units

Inspect the units used by the imported exchanges after unit normalization.


In [ ]:
exchange_units = {
    exchange["unit"]
    for dataset in importer.data
    for exchange in dataset["exchanges"]
}

sorted(exchange_units)

### 6. Map biosphere flows

The SimaPro elementary-flow labels are converted to Brightway-compatible names and categories, followed by a few EXIOBASE-specific corrections.


In [ ]:
importer.randonneur(
    "SimaPro-9-ecoinvent-3.8-biosphere",
    fields=["name", "context", "unit"],
)
importer.randonneur("simapro-9-ecoinvent-3-context")
importer.normalize_labels_to_brightway_standard()
importer.apply_strategy(bi.strategies.biosphere.drop_unspecified_subcategories)
importer.randonneur("SimaPro-9-ecoinvent-3.8-biosphere-manual-matches")
importer.apply_strategy(tweak3)

### 7. Link technosphere and biosphere exchanges

Technosphere exchanges are linked internally within EXIOBASE. Biosphere exchanges are linked to the configured biosphere database.


In [ ]:
importer.match_database(
    fields=["name", "unit", "location"]
)

importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["name", "unit", "categories"],
)
importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["code"],
)
importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["name", "categories", "unit"],
)

importer.statistics()

### 8. Inspect unlinked exchanges

Check which exchanges could not yet be linked to the configured biosphere database. These remaining flows will be placed in a separate EXIOBASE biosphere database.


In [ ]:
pd.DataFrame(importer.unlinked)

### 9. Create an extra biosphere database for remaining flows

Any elementary flows that cannot be linked to the configured biosphere database are stored in a separate EXIOBASE biosphere database.

In [ ]:
if Config.EXIOBASE_EXTRA_BIOSPHERE_NAME not in bd.databases:
    importer.create_new_biosphere(Config.EXIOBASE_EXTRA_BIOSPHERE_NAME)

importer.match_database(
    db_name=Config.EXIOBASE_EXTRA_BIOSPHERE_NAME,
    fields=["name", "unit", "categories"],
)

### 10. Check that all exchanges are linked

Before writing the database, verify that no exchanges remain unlinked.

In [ ]:
pd.DataFrame(importer.unlinked)

### 11. Write the EXIOBASE database

This writes the processed datasets to the current Brightway project.

In [ ]:
importer.write_database()

### 12. Verify the import

In [ ]:
bd.databases